In [1]:
!pip install ultralytics --quiet

import os
import cv2
import yaml
import shutil
import numpy as np
from tqdm import tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.9 MB/s eta 0:00:00a 0:00:01


In [2]:


# Automatically scan Kaggle inputs to find where 'ImageSets' actually is
found_path = None
for root, dirs, files in os.walk("/kaggle/input/"):
    if "ImageSets" in dirs:
        found_path = root
        break

if found_path:
    INPUT_DIR = found_path
    print(f"🎉 Success! Found dataset root at: {INPUT_DIR}")
else:
    print("❌ Could not find ImageSets. Did you add the dataset to the notebook sidebar?")

# Target directory for YOLO format remains the same
OUTPUT_DIR = "/kaggle/working/yolo_food_seg"

for split in ['train', 'val']:
    os.makedirs(os.path.join(OUTPUT_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, 'labels', split), exist_ok=True)

🎉 Success! Found dataset root at: /kaggle/input/datasets/ggrill/foodseg103/FoodSeg103


In [3]:
def process_split(split_name, text_file):
    # Read image IDs from the dataset's text files
    with open(os.path.join(INPUT_DIR, 'ImageSets', text_file), 'r') as f:
        img_ids = [line.strip() for line in f.readlines() if line.strip()]
        
    yolo_split = 'train' if split_name == 'train' else 'val'
    
    print(f"Processing {split_name} split...")
    for img_id in tqdm(img_ids):
        # Paths
        img_path = os.path.join(INPUT_DIR, 'Images', 'img_dir', split_name, f"{img_id}.jpg")
        mask_path = os.path.join(INPUT_DIR, 'Images', 'ann_dir', split_name, f"{img_id}.png")
        
        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue
            
        # Copy image to writable space
        shutil.copy(img_path, os.path.join(OUTPUT_DIR, 'images', yolo_split, f"{img_id}.jpg"))
        
        # Process Mask
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        h, w = mask.shape
        classes = np.unique(mask)
        
        label_lines = []
        for cls in classes:
            if cls == 0: continue # Skip background
            
            # Extract boundaries for this class
            class_mask = (mask == cls).astype(np.uint8) * 255
            contours, _ = cv2.findContours(class_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            for contour in contours:
                if len(contour) < 3: continue
                polygon = contour.reshape(-1, 2)
                normalized = [f"{x/w} {y/h}" for x, y in polygon]
                # Note: YOLO class IDs are 0-indexed, so we subtract 1 if classes start from 1
                label_lines.append(f"{int(cls)-1} " + " ".join(normalized))
                
        # Write the txt label file
        with open(os.path.join(OUTPUT_DIR, 'labels', yolo_split, f"{img_id}.txt"), 'w') as lf:
            lf.write("\n".join(label_lines))

# Run conversion
process_split('train', 'train.txt')
process_split('test', 'test.txt') # treating test split as validation

Processing train split...


100%|██████████| 4983/4983 [00:03<00:00, 1322.80it/s]


Processing test split...


100%|██████████| 2135/2135 [00:01<00:00, 1380.93it/s]


In [4]:
# Read classes
classes_dict = {}
with open(os.path.join(INPUT_DIR, 'category_id.txt'), 'r') as f:
    for line in f.readlines():
        if line.strip():
            parts = line.strip().split('\t') # or split(' ') depending on format
            if len(parts) >= 2:
                # Aligning to 0-indexed values
                classes_dict[int(parts[0])-1] = parts[1]

# Construct YAML structure
yaml_data = {
    'path': OUTPUT_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'names': classes_dict
}

# Save configuration file
with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False)
    
print("Data preparation complete!")

Data preparation complete!


In [6]:
import os
import cv2
import shutil
import numpy as np
from tqdm import tqdm

# Automatically locate the ImageSets folder dynamically
found_path = None
for root, dirs, files in os.walk("/kaggle/input/"):
    if "ImageSets" in dirs:
        found_path = root
        break

INPUT_DIR = found_path
OUTPUT_DIR = "/kaggle/working/yolo_food_seg"

print(f"Dataset root identified at: {INPUT_DIR}")

def process_split_fixed(split_name, text_file):
    # Read image IDs from the dataset's text files
    txt_path = os.path.join(INPUT_DIR, 'ImageSets', text_file)
    with open(txt_path, 'r') as f:
        # Strip trailing newlines/spaces, and ignore extensions if they were baked into the text file
        img_ids = [line.strip().split('.')[0] for line in f.readlines() if line.strip()]
        
    yolo_split = 'train' if split_name == 'train' else 'val'
    
    print(f"Processing {split_name} split ({len(img_ids)} potential files)...")
    copied_count = 0
    
    for img_id in tqdm(img_ids):
        # Build paths exactly matching standard FoodSeg103 layout
        img_path = os.path.join(INPUT_DIR, 'Images', 'img_dir', split_name, f"{img_id}.jpg")
        mask_path = os.path.join(INPUT_DIR, 'Images', 'ann_dir', split_name, f"{img_id}.png")
        
        # If it's not found, try without sub-folders (some Kaggle uploads flatten 'img_dir'/'ann_dir')
        if not os.path.exists(img_path):
            img_path = os.path.join(INPUT_DIR, 'Images', split_name, f"{img_id}.jpg")
            mask_path = os.path.join(INPUT_DIR, 'Images', split_name, f"{img_id}.png")
            
        if not os.path.exists(img_path) or not os.path.exists(mask_path):
            continue
            
        # Copy image to writable space
        shutil.copy(img_path, os.path.join(OUTPUT_DIR, 'images', yolo_split, f"{img_id}.jpg"))
        copied_count += 1
        
        # Process Mask
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        h, w = mask.shape
        classes = np.unique(mask)
        
        label_lines = []
        for cls in classes:
            if cls == 0: continue # Skip background
            
            class_mask = (mask == cls).astype(np.uint8) * 255
            contours, _ = cv2.findContours(class_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            for contour in contours:
                if len(contour) < 3: continue
                polygon = contour.reshape(-1, 2)
                normalized = [f"{x/w} {y/h}" for x, y in polygon]
                label_lines.append(f"{int(cls)-1} " + " ".join(normalized))
                
        # Write the txt label file
        with open(os.path.join(OUTPUT_DIR, 'labels', yolo_split, f"{img_id}.txt"), 'w') as lf:
            lf.write("\n".join(label_lines))
            
    print(f"Successfully processed and copied {copied_count} files for {yolo_split}!")

# Run conversion
process_split_fixed('train', 'train.txt')
process_split_fixed('test', 'test.txt')

Dataset root identified at: /kaggle/input/datasets/ggrill/foodseg103/FoodSeg103
Processing train split (4983 potential files)...


100%|██████████| 4983/4983 [04:39<00:00, 17.85it/s]


Successfully processed and copied 4983 files for train!
Processing test split (2135 potential files)...


100%|██████████| 2135/2135 [01:08<00:00, 31.01it/s]

Successfully processed and copied 2135 files for val!


In [7]:
from ultralytics import YOLO

model = YOLO('yolov8n-seg.pt') 

# Train
results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=20, 
    imgsz=640, 
    batch=16, 
    device=0 
)

Ultralytics 8.4.93 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask

In [8]:
import pandas as pd

In [9]:
results = pd.read_csv("/kaggle/working/runs/segment/train-2/results.csv")

In [11]:
results

,epoch,time,train/box_loss,train/seg_loss,train/cls_loss,train/dfl_loss,train/sem_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),...,metrics/mAP50(M),metrics/mAP50-95(M),val/box_loss,val/seg_loss,val/cls_loss,val/dfl_loss,val/sem_loss,lr/pg0,lr/pg1,lr/pg2
0,1,175.784,0.95612,2.59159,4.80536,1.27927,0,0.00742,0.15539,0.00815,...,0.00818,0.00564,0.93999,2.15785,4.39303,1.31453,0,0.000031,0.000031,0.000031
1,2,347.198,0.91188,2.03080,3.98800,1.27202,0,0.57989,0.06570,0.03646,...,0.03723,0.02681,0.97231,1.92579,3.69709,1.34248,0,0.000059,0.000059,0.000059
2,3,519.531,0.91588,1.91305,3.46899,1.27268,0,0.53528,0.10568,0.06396,...,0.06497,0.04778,0.94539,1.84886,3.34399,1.31823,0,0.000084,0.000084,0.000084
3,4,687.956,0.88897,1.85144,3.13877,1.25474,0,0.50668,0.12198,0.08625,...,0.08782,0.06453,0.91702,1.80576,3.08652,1.28654,0,0.000079,0.000079,0.000079
4,5,859.569,0.86238,1.79907,2.93125,1.23388,0,0.53012,0.13495,0.09855,...,0.10057,0.07434,0.89337,1.74775,2.92573,1.26311,0,0.000075,0.000075,0.000075
5,6,1031.080,0.83232,1.76612,2.77288,1.21211,0,0.50014,0.14731,0.11104,...,0.11267,0.08372,0.86969,1.72871,2.80695,1.24181,0,0.000070,0.000070,0.000070
6,7,1201.810,0.81378,1.73822,2.68353,1.19567,0,0.51641,0.15673,0.12178,...,0.12277,0.09184,0.84808,1.70887,2.73638,1.22048,0,0.000065,0.000065,0.000065
7,8,1372.130,0.79451,1.69940,2.60425,1.17704,0,0.51740,0.16325,0.13375,...,0.13540,0.10141,0.84775,1.68058,2.64629,1.22124,0,0.000061,0.000061,0.000061
8,9,1542.710,0.78955,1.68432,2.53086,1.17389,0,0.52623,0.17653,0.14299,...,0.14480,0.10801,0.84311,1.67405,2.58388,1.21384,0,0.000056,0.000056,0.000056
9,10,1714.240,0.78079,1.67371,2.48433,1.16464,0,0.53363,0.17855,0.14864,...,0.14994,0.11242,0.83736,1.67330,2.56283,1.20975,0,0.000052,0.000052,0.000052


In [13]:
import os
import requests
from ultralytics import YOLO

# 1. Load your model from train2
model = YOLO('/kaggle/working/runs/segment/train-2/weights/last.pt')

# 2. Setup USDA API credentials (replace with your key or use DEMO_KEY)
API_KEY = "4dYGNVmU7KZ8d4Zg8p5A6hizDJai3gPMVZIPCNP6" 
USDA_URL = f"https://api.nal.usda.gov/fdc/v1/foods/search?api_key={API_KEY}"

# Find a real image dynamically from your validation folder
val_img_dir = "/kaggle/working/yolo_food_seg/images/val"
available_images = [f for f in os.listdir(val_img_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]

if not available_images:
    print(" No images found in the validation directory! Check your dataset steps.")
else:
    # Pick the first actual image file found
    test_image_path = os.path.join(val_img_dir, available_images[0])
    print(f"📸 Running inference on a real image: {test_image_path}")

    def get_nutrition_info(food_name):
        payload = {
            "query": food_name,
            "dataType": ["Foundation", "Survey (FNDDS)"],
            "pageSize": 1
        }
        try:
            response = requests.post(USDA_URL, json=payload)
            if response.status_code == 200:
                data = response.json()
                if data.get('foods'):
                    food_item = data['foods'][0]
                    print(f"\n USDA Match: {food_item['description']}")
                    print("-" * 40)
                    for nutrient in food_item.get('foodNutrients', []):
                        if nutrient['nutrientName'] in ['Protein', 'Energy', 'Carbohydrate, by difference', 'Total lipid (fat)']:
                            name = nutrient['nutrientName'].split(',')[0]
                            amount = nutrient['value']
                            unit = nutrient['unitName'].lower()
                            print(f" • {name}: {amount} {unit} (per 100g)")
                else:
                    print(f" No nutritional data found for '{food_name}' in USDA database.")
            else:
                print(f" API Error {response.status_code}: Could not fetch data.")
        except Exception as e:
            print(f"An error occurred: {e}")

    # 3. Run Inference
    results = model(test_image_path)

    # 4. Extract detected food classes and call API
    detected_classes = set()
    for r in results:
        for box in r.boxes:
            class_id = int(box.cls)
            class_name = model.names[class_id]
            detected_classes.add(class_name)

    print(f" Model Detected: {list(detected_classes)}")

    for food in detected_classes:
        get_nutrition_info(food)

📸 Running inference on a real image: /kaggle/working/yolo_food_seg/images/val/00007029.jpg

image 1/1 /kaggle/working/yolo_food_seg/images/val/00007029.jpg: 640x480 1 coffee, 1 tomato, 9.3ms
Speed: 2.0ms preprocess, 9.3ms inference, 3.3ms postprocess per image at shape (1, 3, 640, 480)
🥗 Model Detected: ['coffee', 'tomato']

🍏 USDA Match: Coffee, brewed
----------------------------------------
 • Protein: 0.12 g (per 100g)
 • Total lipid (fat): 0.02 g (per 100g)
 • Carbohydrate: 0 g (per 100g)
 • Energy: 1 kcal (per 100g)

🍏 USDA Match: Tomatoes, raw
----------------------------------------
 • Protein: 0.82 g (per 100g)
 • Total lipid (fat): 0.31 g (per 100g)
 • Carbohydrate: 4.04 g (per 100g)
 • Energy: 20 kcal (per 100g)


In [15]:
import os
import cv2
import requests
import numpy as np
from ultralytics import YOLO

# ==========================================
# 1. INITIALIZATION & DATASET PREPARATION
# ==========================================
# Load your trained model weights (pointing to your train2 check point folder)
model = YOLO('/kaggle/working/runs/segment/train-2/weights/last.pt')

# USDA API Setup (Replace with your actual API key for high rate limits)
USDA_API_KEY = "4dYGNVmU7KZ8d4Zg8p5A6hizDJai3gPMVZIPCNP6" 
USDA_URL = f"https://api.nal.usda.gov/fdc/v1/foods/search?api_key={USDA_API_KEY}"

# HEURISTIC PROXIES configuration 
# Define which foods from FoodSeg103 are countable vs area-based, alongside their assumptions
FOOD_CONFIG = {
    # Countable Foods: mapped to fixed average weights (grams per instance)
    "countable": {
        "banana": 120,
        "apple": 180,
        "egg": 50,
        "tomato": 100,
        "orange": 130
    },
    # Area-Based Foods: mapped to a multiplier conversion factor (grams per pixel)
    # Note: Adjust conversion factor based on camera height/resolution calibration
    "area_based": {
        "rice": 0.005,
        "curry": 0.006,
        "noodles": 0.004,
        "bread": 0.03,
        "coffee": 0.002
    },
    # Default fallback for unmapped foods
    "default_weight": 150 
}

# ==========================================
# 2. IMAGE PREPROCESSING & DETECTION
# ==========================================
# Target dynamic validation image from earlier
IMAGE_PATH = "/kaggle/working/yolo_food_seg/images/val/00007029.jpg"

if not os.path.exists(IMAGE_PATH):
    raise FileNotFoundError(f"Target test image not found at {IMAGE_PATH}")

# Run YOLOv8-Seg Inference
results = model(IMAGE_PATH)[0]
img_h, img_w, _ = cv2.imread(IMAGE_PATH).shape

print("\n--- Food Detection & Quantity Estimation ---")

meal_summary = []

# ==========================================
# 3. MASK PROCESSING & QUANTITY ESTIMATION
# ==========================================
if results.masks is not None:
    # Iterate over every single detection layer
    for i, (box, mask_obj) in enumerate(zip(results.boxes, results.masks)):
        class_id = int(box.cls[0])
        class_name = model.names[class_id].lower()
        
        # Extract binary mask array (0 or 255) resized to original image layout
        mask = mask_obj.data[0].cpu().numpy()
        binary_mask = (mask > 0.5).astype(np.uint8) * 255
        binary_mask = cv2.resize(binary_mask, (img_w, img_h), interpolation=cv2.INTER_NEAREST)
        
        estimated_weight = 0.0
        calculation_method = ""

        # Strategy A: Countable Item Heuristic using Connected Components
        if class_name in FOOD_CONFIG["countable"]:
            # Perform connected component analysis to separate disjoint instances
            num_labels, labels_im = cv2.connectedComponents(binary_mask)
            # The function returns background as label 0, so subtract 1 to get object instances
            instance_count = max(1, num_labels - 1) 
            
            unit_weight = FOOD_CONFIG["countable"][class_name]
            estimated_weight = instance_count * unit_weight
            calculation_method = f"Countable ({instance_count} instance(s) × {unit_weight}g)"
            
        # Strategy B: Region-Based Food Heuristic using Pixel Mask Area
        elif class_name in FOOD_CONFIG["area_based"]:
            # Compute total active spatial area covered by mask pixels
            pixel_area = np.sum(binary_mask == 255)
            multiplier = FOOD_CONFIG["area_based"][class_name]
            estimated_weight = pixel_area * multiplier
            calculation_method = f"Area-based ({pixel_area} px × factor {multiplier})"
            
        # Strategy C: Global Fallback Heuristic
        else:
            estimated_weight = FOOD_CONFIG["default_weight"]
            calculation_method = f"Default allocation (Unmapped food item type)"
            
        # Ensure minimum rounding safety floor
        estimated_weight = round(max(10.0, estimated_weight), 1)
        
        print(f" Detected: {class_name.capitalize()}")
        print(f"   ↳ Method: {calculation_method}")
        print(f"   ↳ Est. Weight: {estimated_weight}g")
        
        meal_summary.append({
            "name": class_name,
            "weight": estimated_weight
        })
else:
    print("No items segmented by the model in this image.")

# ==========================================
# 4. NUTRITION LOOKUP & CALORIE COMPUTATION
# ==========================================
print("\n--- Live USDA Nutritional Integration & Calorie Output ---")
total_meal_calories = 0.0

def fetch_usda_calories_per_100g(food_query):
    """Retrieves standard kcal content per 100 grams from USDA API"""
    payload = {
        "query": food_query,
        "dataType": ["Foundation", "Survey (FNDDS)"],
        "pageSize": 1
    }
    try:
        response = requests.post(USDA_URL, json=payload)
        if response.status_code == 200:
            data = response.json()
            if data.get('foods'):
                matched_food = data['foods'][0]
                # Look specifically for standard Energy element measurements
                for nutrient in matched_food.get('foodNutrients', []):
                    if nutrient.get('nutrientName') == 'Energy' and nutrient.get('unitName') == 'KCAL':
                        return float(nutrient['value']), matched_food['description']
        return None, None
    except Exception as e:
        print(f" API Request error parsing '{food_query}': {e}")
        return None, None

# Calculate Individual Component Contributions and Total Calories
for item in meal_summary:
    kcal_per_100g, official_name = fetch_usda_calories_per_100g(item["name"])
    
    # Fallback to general generic standard values if API yields empty fields
    if kcal_per_100g is None:
        kcal_per_100g = 150.0  
        official_name = f"{item['name']} (Standard Generic Approximation)"
        print(f" No exact USDA match found for '{item['name']}'. Defaulting to standard reference scale.")
        
    # Calorie Equation: (Estimated Weight / 100g) * Base Calories per 100g
    item_calories = (item["weight"] / 100.0) * kcal_per_100g
    item_calories = round(item_calories, 1)
    total_meal_calories += item_calories
    
    print(f"\n Item: {official_name}")
    print(f"   • Portion Weight  : {item['weight']}g")
    print(f"   • Energy Density  : {kcal_per_100g} kcal / 100g")
    print(f"   • Net Contribution: {item_calories} kcal")

print("\n" + "="*45)
print(f"  TOTAL MEAL ENERGY: {round(total_meal_calories, 1)} kcal")
print("="*45)


image 1/1 /kaggle/working/yolo_food_seg/images/val/00007029.jpg: 640x480 1 coffee, 1 tomato, 9.5ms
Speed: 2.6ms preprocess, 9.5ms inference, 3.2ms postprocess per image at shape (1, 3, 640, 480)

--- Food Detection & Quantity Estimation ---
 Detected: Coffee
   ↳ Method: Area-based (17222 px × factor 0.002)
   ↳ Est. Weight: 34.4g
 Detected: Tomato
   ↳ Method: Countable (1 instance(s) × 100g)
   ↳ Est. Weight: 100g

--- Live USDA Nutritional Integration & Calorie Output ---

🥑 Item: Coffee, brewed
   • Portion Weight  : 34.4g
   • Energy Density  : 1.0 kcal / 100g
   • Net Contribution: 0.3 kcal

🥑 Item: Tomatoes, raw
   • Portion Weight  : 100g
   • Energy Density  : 20.0 kcal / 100g
   • Net Contribution: 20.0 kcal

 🔥 TOTAL MEAL ENERGY: 20.3 kcal
